# Datacleaning
Cleaning and structuring a messy employee dataset (`raw_data.csv`) with **pandas**.

Steps: load → inspect → clean text → fix types → handle invalid & missing values → dedupe → structure → export.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
print('pandas', pd.__version__)

## 2. Load the raw data

In [ ]:
df = pd.read_csv('raw_data.csv')
df.head(10)

## 3. Inspect the data

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
print('Missing values per column:')
df.isna().sum()

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
for col in ['city', 'department', 'is_active']:
    print(f'\n{col}:', df[col].unique())

## 4. Remove duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicates -> {len(df)} rows')

## 5. Clean text columns
Trim spaces, fix capitalisation, and standardise city names.

In [ ]:
df['name'] = df['name'].str.strip().str.title()
df['department'] = df['department'].str.strip().str.title().replace({'Hr': 'HR'})

city_map = {'bengaluru': 'Bangalore', 'new delhi': 'Delhi'}
df['city'] = (df['city'].str.strip().str.lower()
              .replace(city_map)
              .str.title()
              .replace('', np.nan))
df[['name', 'city', 'department']].drop_duplicates().head(10)

## 6. Fix numeric columns
- `age`: convert text like `'thirty'` to NaN, drop impossible values (<18 or >70)
- `salary`: strip `₹` and commas, convert to number, flag outliers

In [ ]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[~df['age'].between(18, 70), 'age'] = np.nan

df['salary'] = (df['salary'].astype(str)
                .str.replace(r'[₹,\s]', '', regex=True)
                .replace({'nan': np.nan, '': np.nan}))
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df.loc[df['salary'] > 500000, 'salary'] = np.nan   # unrealistic outlier
df[['age', 'salary']].describe()

## 7. Parse dates
The raw column mixes formats (`2023-01-15`, `15/02/2023`, `2023.03.20`, `March 5 2023`).

In [ ]:
def parse_date(x):
    if pd.isna(x) or str(x).strip() == '':
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%Y.%m.%d', '%B %d %Y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['join_date'] = df['join_date'].apply(parse_date)
df['join_date'].value_counts(dropna=False)

## 8. Validate emails and standardise booleans

In [ ]:
valid = df['email'].str.match(r'^[\w.+-]+@[\w-]+\.[\w.]+$', na=False)
df.loc[~valid, 'email'] = np.nan

bool_map = {'y': True, 'yes': True, '1': True, 'n': False, 'no': False, '0': False}
df['is_active'] = df['is_active'].astype(str).str.strip().str.lower().map(bool_map)
df[['email', 'is_active']].head()

## 9. Handle missing values
- Numeric → fill with median (by department where possible)
- Categorical → `'Unknown'`
- Dates / emails → keep as missing (can't be guessed)

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())
df['salary'] = df.groupby('department')['salary'].transform(lambda s: s.fillna(s.median()))
df['salary'] = df['salary'].fillna(df['salary'].median())

df['city'] = df['city'].fillna('Unknown')
df['department'] = df['department'].fillna('Unknown')
df['is_active'] = df['is_active'].fillna(False)

df.isna().sum()

## 10. Structure the data
Set correct dtypes, add derived columns, reorder and sort.

In [ ]:
df = df.astype({'id': 'int64', 'age': 'int64', 'salary': 'float64', 'is_active': 'bool',
                'city': 'category', 'department': 'category'})

df['salary_band'] = pd.cut(df['salary'], bins=[0, 50000, 100000, np.inf],
                           labels=['Low', 'Medium', 'High'])
df['join_year'] = df['join_date'].dt.year.astype('Int64')

df = df[['id', 'name', 'age', 'city', 'department', 'salary', 'salary_band',
         'join_date', 'join_year', 'email', 'is_active']]
df = df.sort_values('id').reset_index(drop=True)
df.info()
df.head(10)

## 11. Quick summary

In [ ]:
summary = (df.groupby('department', observed=True)
             .agg(employees=('id', 'count'),
                  avg_age=('age', 'mean'),
                  avg_salary=('salary', 'mean'),
                  active=('is_active', 'sum'))
             .round(1))
summary

## 12. Export clean data

In [ ]:
df.to_csv('clean_data.csv', index=False)
summary.to_csv('department_summary.csv')
print('Saved clean_data.csv', df.shape, 'and department_summary.csv')